# Inspect broken math segments in translations

For each translation config, shows which math segments from the source are missing or changed in the translation.

In [6]:
import json
import re
from pathlib import Path

ROOT = Path("..")

# $$...$$ matched before $...$ to avoid splitting double-dollar signs.
# (?!\d+[,\s]) excludes currency amounts like $1,000 and $100 (digit then comma/space)
# while still allowing math starting with a digit like $3x+2$.
MATH_RE = re.compile(r'\$\$[\s\S]*?\$\$|\$(?!\d+[,\s])[^$\n]+?\$')

def load_jsonl(path):
    with open(path) as f:
        return {r["idx"]: r for line in f if (r := json.loads(line.strip()))}

dataset = load_jsonl(ROOT / "data" / "ready_dataset.jsonl")
print(f"Dataset: {len(dataset)} examples")

Dataset: 5000 examples


In [7]:
# Load all available translation configs
trans_dir = ROOT / "eval_results" / "translations"
configs = {p.stem: load_jsonl(p) for p in sorted(trans_dir.glob("*.jsonl"))}
print("Available configs:", list(configs.keys()))

Available configs: ['api_glos', 'api_no_glos']


In [8]:
import re as _re

def normalize(text):
    text = text.replace(r"\_", "_")
    # add braces to bare multi-character subscripts: a_11 -> a_{11}
    text = _re.sub(r'_(\w{2,})', r'_{\1}', text)
    return text

def find_broken(translations, dataset, field):
    """Return list of dicts describing each broken math segment."""
    broken = []
    for idx, t in sorted(translations.items()):
        src = dataset[idx][field]
        tgt = t[f"{field}_pl"]
        tgt_norm = normalize(tgt)
        for seg in MATH_RE.findall(src):
            if normalize(seg) not in tgt_norm:
                broken.append({"idx": idx, "field": field, "segment": seg})
    return broken

In [9]:
# Print broken segments for all configs
for cfg, translations in configs.items():
    print(f"\n{'='*60}")
    print(f"Config: {cfg}  ({len(translations)} examples)")
    print(f"{'='*60}")

    for field in ["problem", "solution"]:
        broken = find_broken(translations, dataset, field)
        if not broken:
            print(f"  [{field}] No broken segments.")
            continue
        print(f"  [{field}] {len(broken)} broken segment(s):")
        for b in broken:
            print(f"    idx={b['idx']}  {b['segment'][:80]}")


Config: api_glos  (5 examples)
  [problem] No broken segments.
  [solution] No broken segments.

Config: api_no_glos  (5 examples)
  [problem] No broken segments.
  [solution] No broken segments.


In [10]:
# Summary: count of broken segments across all configs
from collections import Counter

for cfg, translations in configs.items():
    all_broken = find_broken(translations, dataset, "problem") + find_broken(translations, dataset, "solution")
    total_segs = sum(
        len(MATH_RE.findall(normalize(dataset[idx][f])))
        for idx in translations
        for f in ["problem", "solution"]
    )
    print(f"{cfg}: {len(all_broken)}/{total_segs} broken")

api_glos: 0/51 broken
api_no_glos: 0/51 broken
